# Module 3 - Class 6: End-to-End Olist Data Prep

A full data-preparation pipeline that builds the Olist modeling table.


---

## Stage 1 - Load (10 min)


In [ ]:
# ---.
import pandas as pd, numpy as np

orders   = pd.read_csv('olist_data/olist_orders_dataset.csv')
customers = pd.read_csv('olist_data/olist_customers_dataset.csv')
items    = pd.read_csv('olist_data/olist_order_items_dataset.csv')
payments = pd.read_csv('olist_data/olist_order_payments_dataset.csv')
reviews  = pd.read_csv('olist_data/olist_order_reviews_dataset.csv')
products = pd.read_csv('olist_data/olist_products_dataset.csv')
sellers  = pd.read_csv('olist_data/olist_sellers_dataset.csv')
geo      = pd.read_csv('olist_data/olist_geolocation_dataset.csv')

for name, t in [('orders', orders), ('customers', customers), ('items', items),
                ('payments', payments), ('reviews', reviews),
                ('products', products), ('sellers', sellers)]:
    print(f'{name:10s}: {t.shape}')


## Stage 2 - Clean orders (15 min)


In [ ]:
# Clean order dates and filter modellable rows.
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors='coerce')

MODEL_STATUSES = {'delivered', 'invoiced', 'shipped', 'approved'}
orders_model = orders[orders['order_status'].isin(MODEL_STATUSES)].copy()
orders_undelivered = orders[~orders.index.isin(orders_model.index)].copy()
orders_undelivered.to_parquet('orders_undelivered.parquet', index=False)

before = len(orders_model)
orders_model = orders_model.dropna(subset=['order_delivered_customer_date',
                                          'order_estimated_delivery_date'])
print(f'Modellable rows: {len(orders_model):,} (dropped {before - len(orders_model):,})')


## Stage 3 - Aggregate items + payments (15 min)


In [ ]:
# Aggregate order items and payment features.
order_totals = items.groupby('order_id').agg(
    num_items=('order_item_id', 'count'),
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
).reset_index()

primary_pay = (payments
    .sort_values('payment_value', ascending=False)
    .drop_duplicates('order_id')[['order_id', 'payment_type', 'payment_installments']])

first_seller = (items.sort_values(['order_id', 'order_item_id'])
                     .drop_duplicates('order_id')[['order_id', 'seller_id']])


## Stage 4 - Geolocation merge + Haversine (20 min)


In [ ]:
# Prepare location averages and distance logic.
R = 6371
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

geo_lookup = geo.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'mean'),
    lon=('geolocation_lng', 'mean'),
).reset_index()


## Stage 5 - Master frame: merge everything + engineer all features (15 min)


In [ ]:
# Build the master frame from orders_model
df = orders_model.merge(customers[['customer_id', 'customer_unique_id',
                                   'customer_state', 'customer_zip_code_prefix']],
                        on='customer_id', how='left')
df = df.merge(order_totals, on='order_id', how='left')
df = df.merge(primary_pay, on='order_id', how='left')
df = df.merge(first_seller, on='order_id', how='left')
df = df.merge(sellers[['seller_id', 'seller_state', 'seller_zip_code_prefix']],
              on='seller_id', how='left')

# Customer lat/lon
df = df.merge(geo_lookup.rename(columns={'lat':'cust_lat', 'lon':'cust_lon'}),
              left_on='customer_zip_code_prefix',
              right_on='geolocation_zip_code_prefix', how='left')
df = df.drop(columns=['geolocation_zip_code_prefix'])

# Seller lat/lon
df = df.merge(geo_lookup.rename(columns={'lat':'sell_lat','lon':'sell_lon',
                                         'geolocation_zip_code_prefix':'sz'}),
              left_on='seller_zip_code_prefix', right_on='sz', how='left')
df = df.drop(columns=['sz'])

# Distance
df['distance_km'] = haversine(df['cust_lat'].values, df['cust_lon'].values,
                              df['sell_lat'].values, df['sell_lon'].values)

# Date features
df['delivery_days']      = (df['order_delivered_customer_date']
                            - df['order_purchase_timestamp']).dt.days
df['estimate_days']      = (df['order_estimated_delivery_date']
                            - df['order_purchase_timestamp']).dt.days
df['is_late']            = (df['order_delivered_customer_date']
                            > df['order_estimated_delivery_date']).astype(int)
df['purchase_year']      = df['order_purchase_timestamp'].dt.year
df['purchase_month']     = df['order_purchase_timestamp'].dt.month
df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek
df['purchase_hour']      = df['order_purchase_timestamp'].dt.hour
df['is_weekend']         = (df['purchase_dayofweek'] >= 5).astype(int)
df['log_freight']        = np.log1p(df['total_freight'].fillna(0))

print(f'Master frame: {df.shape}')


## Stage 6 - Add reviews + save (10 min)


In [ ]:
# Add review data and save the final parquet file.
reviews_first = (reviews.sort_values(['order_id', 'review_creation_date'])
                          .drop_duplicates('order_id')
                          [['order_id', 'review_score', 'review_comment_message']])
df = df.merge(reviews_first, on='order_id', how='left')

FINAL_COLS = [
    'order_id', 'customer_unique_id', 'customer_state', 'seller_state',
    'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour',
    'is_weekend',
    'num_items', 'total_price', 'total_freight', 'log_freight',
    'payment_type', 'payment_installments',
    'distance_km', 'delivery_days', 'estimate_days', 'is_late',
    'review_score', 'review_comment_message',
]
out = df[FINAL_COLS].copy()
out.to_parquet('olist_clean.parquet', index=False)

print(f'✅ olist_clean.parquet: {out.shape}')
print(f'   is_late rate:    {out["is_late"].mean():.4f} (expect ~0.07)')
print(f'   reviews present: {out["review_score"].notna().mean():.3f}')
print(f'\nColumns: {list(out.columns)}')


## Findings

The final save cell prints the row count, late-delivery rate, review coverage, and schema after `olist_clean.parquet` is written.

Cleaning notes: undelivered statuses are separated, rows missing delivery or estimate dates are removed from the modeling frame, and order-level totals, primary payment, seller state, distance, and first review are merged into one table.

Question for the M4 mentor: should late-delivery models prioritize recall even if that creates more false alerts for operations?
